# Project 10: GNN-Based Credit Contagion

Graph neural networks for modeling systemic risk and default contagion in financial networks.

This notebook walks through:
1. **Network Generation** -- Constructing a synthetic interbank network and computing centrality metrics
2. **Eisenberg-Noe Cascade** -- Running the clearing mechanism from different shock points
3. **DebtRank Analysis** -- Identifying systemically important institutions
4. **GCN Training** -- Training a graph convolutional network to predict defaults
5. **Comparison** -- GCN predictions vs centrality-based rankings

All implementations are pure numpy (no PyTorch).

In [ ]:
import sys
from pathlib import Path

# Add src to path
src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from contagion import eisenberg_noe_clearing, simulate_cascade, systemic_importance
from diagnostics import (
    plot_cascade_size,
    plot_debtrank_distribution,
    plot_gcn_predictions,
    plot_network,
)
from gcn import GCN
from network import compute_centrality, generate_financial_network, network_stats
from trainer import evaluate_gcn, train_gcn

SEED = 42
print("Imports OK")

## 1. Generate Financial Network

We generate a synthetic interbank network with 50 institutions and 150 directed exposures.
Each edge represents a liability from one bank to another, weighted by exposure amount.

Node features capture each institution's financial health:
- **Capital ratio**: (assets - liabilities) / assets
- **Leverage**: total liabilities / equity
- **Asset quality**: fraction of performing claims
- **Size**: log-normalised total assets
- **Interbank exposure ratio**: interbank claims / total assets

In [ ]:
# Generate the network
net = generate_financial_network(n_nodes=50, n_edges=150, seed=SEED)

print(f"Adjacency shape: {net['adjacency'].shape}")
print(f"Node features shape: {net['node_features'].shape}")
print(f"Labels (capital ratio based): {int(net['labels'].sum())} defaults out of 50")

# Network statistics
stats = network_stats(net["adjacency"])
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Compute centrality metrics
centrality_df = compute_centrality(net["adjacency"])
print("\nTop 10 nodes by degree centrality:")
print(centrality_df.sort_values("degree_centrality", ascending=False).head(10).to_string(index=False))

In [ ]:
# Visualize the network
fig, ax = plot_network(
    net["adjacency"],
    net["node_features"][:, 0],  # Color by capital ratio
    title="Interbank Network (colored by capital ratio)"
)
plt.show()

## 2. Eisenberg-Noe Cascade Simulation

The Eisenberg-Noe (2001) model finds the equilibrium clearing payments in an interbank network.
Each bank pays the minimum of its total obligations and its available resources (external assets plus incoming payments).

We simulate cascades by shocking individual nodes (reducing their assets by 50%) and observing how defaults propagate.

In [ ]:
# Run Eisenberg-Noe clearing on the undisturbed network
payments, defaults = eisenberg_noe_clearing(net["liabilities"], net["assets"])
print(f"Equilibrium defaults (no shock): {int(defaults.sum())} out of 50")
print(f"Total payments: {payments.sum():.1f}")
print(f"Total obligations: {net['liabilities'].sum(axis=1).sum():.1f}")

In [ ]:
# Cascade analysis: shock each node individually
cascade_results = []
for node in range(50):
    result = simulate_cascade(
        net["liabilities"], net["assets"],
        shocked_nodes=[node], shock_fraction=0.5
    )
    cascade_results.append({
        "shocked_node": node,
        "n_defaults": result["n_defaults"],
        "total_loss": result["total_loss"],
        "rounds": result["rounds"],
    })

cascade_df = pd.DataFrame(cascade_results)
print("\nCascade statistics:")
print(f"  Mean defaults per shock: {cascade_df['n_defaults'].mean():.1f}")
print(f"  Max defaults: {cascade_df['n_defaults'].max()}")
print(f"  Most dangerous node: {cascade_df.loc[cascade_df['n_defaults'].idxmax(), 'shocked_node']}")

In [ ]:
# Plot cascade sizes
fig, ax = plot_cascade_size(cascade_df)
plt.show()

## 3. DebtRank: Systemic Importance

DebtRank (Battiston et al., 2012) measures the fraction of total economic value lost when a single institution becomes distressed. It uses a linear contagion model where distress propagates proportionally to bilateral exposures.

A node with high DebtRank is "too interconnected to fail" -- its distress would cause significant losses across the system.

In [ ]:
# Compute DebtRank for all nodes
debtrank = systemic_importance(net["liabilities"], net["assets"])

print("DebtRank statistics:")
print(f"  Mean: {debtrank.mean():.4f}")
print(f"  Max:  {debtrank.max():.4f} (node {np.argmax(debtrank)})")
print(f"  Min:  {debtrank.min():.4f} (node {np.argmin(debtrank)})")

# Top 5 systemically important nodes
top5 = np.argsort(-debtrank)[:5]
print("\nTop 5 systemically important nodes:")
for i, node in enumerate(top5):
    print(f"  {i+1}. Node {node}: DebtRank = {debtrank[node]:.4f}")

In [ ]:
# Plot DebtRank distribution
fig, ax = plot_debtrank_distribution(debtrank)
plt.show()

## 4. GCN Training: Predicting Defaults

We train a 2-layer Graph Convolutional Network (Kipf & Welling, 2017) to predict which institutions will default, using the Eisenberg-Noe equilibrium defaults as labels.

The GCN leverages both node features and network structure:
- Layer 1: aggregates neighbor information with ReLU activation
- Layer 2: produces default probabilities with sigmoid activation

Training uses Natural Evolution Strategies (NES) since we have no autograd.

In [ ]:
# Prepare labels: combine capital-ratio defaults with Eisenberg-Noe defaults
combined_labels = np.maximum(net["labels"], defaults)
print(f"Training labels: {int(combined_labels.sum())} defaults out of 50")

# Initialise GCN
gcn = GCN(n_features=5, hidden_dim=16, n_classes=1, seed=SEED)
print(f"GCN parameters: {gcn.num_params()}")

# Train
loss_history = train_gcn(
    gcn=gcn,
    X=net["node_features"],
    A=net["adjacency"],
    y=combined_labels,
    n_epochs=80,
    lr=0.03,
    seed=SEED,
    population_size=40,
)

print(f"\nInitial loss: {loss_history[0]:.4f}")
print(f"Final loss:   {loss_history[-1]:.4f}")

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color="steelblue", linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.set_title("GCN Training Loss")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Evaluate
metrics = evaluate_gcn(gcn, net["node_features"], net["adjacency"], combined_labels)
print("GCN Evaluation Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Plot predictions vs true labels
y_pred = gcn.forward(net["node_features"], net["adjacency"]).ravel()
fig, ax = plot_gcn_predictions(combined_labels, y_pred)
plt.show()

## 5. Comparison: GCN Predictions vs Centrality Rankings

We compare the GCN's predicted default probabilities with traditional centrality-based measures of systemic importance. The key question: does the GCN capture information beyond simple network topology?

In [ ]:
# Build comparison DataFrame
comparison = centrality_df.copy()
comparison["debtrank"] = debtrank
comparison["gcn_default_prob"] = y_pred
comparison["true_label"] = combined_labels
comparison["cascade_defaults"] = cascade_df["n_defaults"].values

print("Top 10 nodes by GCN default probability:")
top_gcn = comparison.sort_values("gcn_default_prob", ascending=False).head(10)
print(top_gcn[["node", "gcn_default_prob", "debtrank", "degree_centrality", "true_label"]].to_string(index=False))

In [ ]:
# Correlation analysis
print("\nCorrelation matrix (key metrics):")
corr_cols = ["degree_centrality", "eigenvector_centrality", "debtrank", "gcn_default_prob", "cascade_defaults"]
corr_matrix = comparison[corr_cols].corr()
print(corr_matrix.round(3).to_string())

In [ ]:
# Scatter: GCN default prob vs DebtRank
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ["red" if l == 1 else "blue" for l in combined_labels]

axes[0].scatter(comparison["debtrank"], comparison["gcn_default_prob"], c=colors, alpha=0.6, edgecolors="black", linewidths=0.5)
axes[0].set_xlabel("DebtRank")
axes[0].set_ylabel("GCN Default Probability")
axes[0].set_title("GCN vs DebtRank")

axes[1].scatter(comparison["degree_centrality"], comparison["gcn_default_prob"], c=colors, alpha=0.6, edgecolors="black", linewidths=0.5)
axes[1].set_xlabel("Degree Centrality")
axes[1].set_ylabel("GCN Default Probability")
axes[1].set_title("GCN vs Degree Centrality")

axes[2].scatter(comparison["cascade_defaults"], comparison["gcn_default_prob"], c=colors, alpha=0.6, edgecolors="black", linewidths=0.5)
axes[2].set_xlabel("Cascade Defaults (when shocked)")
axes[2].set_ylabel("GCN Default Probability")
axes[2].set_title("GCN vs Cascade Size")

for ax in axes:
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

print("\nRed = true default, Blue = non-default")

In [ ]:
# Visualize network colored by GCN predictions
fig, ax = plot_network(
    net["adjacency"],
    y_pred,
    title="Interbank Network (colored by GCN default probability)"
)
plt.show()